<a href="https://colab.research.google.com/github/rm-dev-1/adk_dev/blob/main/adk_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import importlib.metadata as md
from google.colab import userdata

print("google-genai:", md.version("google-genai"))  # 入っているバージョンを確認
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

google-genai: 3.6.1


In [5]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")  # 個人のGoogleアカウントで認証する
WORKDIR = Path("/content/drive/MyDrive/gemini-learning")
WORKDIR.mkdir(exist_ok=True)

CACHE_PATH = WORKDIR / "cache.json"
print("保存先:", WORKDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
保存先: /content/drive/MyDrive/gemini-learning


In [6]:
TICKETS = [
    {"id": 1, "text": "昨夜の日次バッチが失敗していて、今朝のレポートが空です", "cat": "pipeline_failure", "pri": 1},
    {"id": 2, "text": "売上ダッシュボードの数字が先週から更新されていません", "cat": "dashboard_issue", "pri": 2},
    {"id": 3, "text": "マーケ部の新メンバーにBigQueryの閲覧権限をお願いします", "cat": "access_request", "pri": 3},
    {"id": 4, "text": "顧客テーブルに同じ顧客IDが重複して入っているようです", "cat": "data_quality", "pri": 2},
    {"id": 5, "text": "今月のクエリ料金が先月の3倍になっています。原因を知りたい", "cat": "cost_question", "pri": 2},
    {"id": 6, "text": "ストリーミング取り込みが2時間止まっています。至急対応してください", "cat": "pipeline_failure", "pri": 1},
    {"id": 7, "text": "ダッシュボードのフィルタの使い方を教えてください", "cat": "other", "pri": 3},
    {"id": 8, "text": "注文テーブルの金額列にマイナス値が混ざっています", "cat": "data_quality", "pri": 2},
    {"id": 9, "text": "データ分析用のサービスアカウントの権限を拡張したい", "cat": "access_request", "pri": 3},
    {"id": 10, "text": "ダッシュボードが開けず、エラー画面が出ます。役員会議で使います", "cat": "dashboard_issue", "pri": 1},
]
print(len(TICKETS), "件")

10 件


In [7]:
import json
import re
import time
from enum import Enum

import pandas as pd
from pydantic import BaseModel, Field
from google import genai
from google.genai import types, errors

client = genai.Client()

class Category(str, Enum):
    PIPELINE_FAILURE = "pipeline_failure"
    DASHBOARD_ISSUE = "dashboard_issue"
    ACCESS_REQUEST = "access_request"
    DATA_QUALITY = "data_quality"
    COST_QUESTION = "cost_question"
    OTHER = "other"

class Ticket(BaseModel):
    category: Category
    priority: int = Field(description="1=緊急(業務停止・期限あり), 2=通常, 3=低")
    summary: str = Field(description="30字以内の要約")
    needs_human: bool = Field(description="人間の担当者が対応すべきならtrue")

In [8]:
MODEL = "gemini-3.5-flash"  # 公式ドキュメントで最新のモデル名を確認

PROMPTS = {
    "baseline": """あなたは社内データ基盤のヘルプデスク担当です。
問い合わせを分類してください。判断に迷う場合は、影響範囲が大きい方を優先します。""",

    "A_definitions": """あなたは社内データ基盤のヘルプデスク担当です。
問い合わせを分類してください。

カテゴリ定義:
- pipeline_failure: バッチ・取り込み・ETLなどの処理の失敗、停止、遅延
- dashboard_issue: ダッシュボードの表示障害、数値の不具合の報告(使い方の質問は含まない)
- access_request: 権限の付与・変更・削除の依頼
- data_quality: データの重複、欠損、不整合
- cost_question: 料金、コスト削減に関する質問
- other: 上記以外(使い方の質問、お礼、一般的な問い合わせなど)""",
}

EXPERIMENT = "baseline"   # ← 実験のたびにここを切り替える
SYSTEM = PROMPTS[EXPERIMENT]

In [9]:
MIN_INTERVAL = 13  # 無料枠5回/分 → 12秒+余裕。制限はモデルごとに違うので公式で確認
CACHE = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
_last_call = 0.0

def classify(text: str, system: str, model: str, retries: int = 5) -> Ticket:
    global _last_call
    key = f"{model}|{system}|{text}"
    if key in CACHE:
        return Ticket.model_validate(CACHE[key])

    for attempt in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()

        try:
            resp = client.models.generate_content(
                model=model,
                contents=text,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=Ticket,
                    temperature=0,
                ),
            )
            ticket = Ticket.model_validate_json(resp.text)
            CACHE[key] = ticket.model_dump(mode="json")
            CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False))
            return ticket
        except errors.ClientError as e:
            if e.code == 429:
                m = re.search(r"retry in ([\d.]+)s", str(e))
                time.sleep(float(m.group(1)) + 1 if m else 15)
                continue
            raise
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("リトライ上限に達しました")

In [10]:
rows = []
for i, t in enumerate(TICKETS, 1):
    print(f"{i}/{len(TICKETS)}", end=" ", flush=True)
    r = classify(t["text"], SYSTEM, MODEL)
    rows.append({
        "id": t["id"], "text": t["text"],
        "expected_cat": t["cat"], "expected_pri": t["pri"],
        "pred_cat": r.category.value, "pred_pri": r.priority,
        "summary": r.summary, "needs_human": r.needs_human,
    })

df = pd.DataFrame(rows)
df.to_csv(WORKDIR / f"result_{EXPERIMENT}.csv", index=False)
print("\n完了")

1/10 2/10 

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}